# Gold setup - table DDL (run once)

Creates empty gold tables with primary/foreign keys and comments.
The schema itself (`team_crypto_gold`) is created in `infra_setup.ipynb` together with bronze and silver.

Load notebooks never create or replace tables, so row filters and column masks from `05_governance` stay attached.

Star schema: two facts (`fact_ohlc_hourly`, `fact_price_tick`) sharing three dimensions (`dim_asset`, `dim_date`, `dim_source`).

In [0]:
%run ../config/config

### Dimensions

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD}.dim_date (
    date_key     INT     NOT NULL COMMENT 'Date in yyyyMMdd format, e.g. 20261027',
    full_date    DATE    NOT NULL,
    year         INT,
    quarter      INT,
    month        INT,
    month_name   STRING,
    day_of_month INT,
    day_name     STRING,
    is_weekend   BOOLEAN,
    CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
)
COMMENT 'Calendar dimension, one row per day'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD}.dim_asset (
    asset_key  BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key of the asset',
    symbol     STRING NOT NULL COMMENT 'Business key: BTC, ETH, SOL ...',
    asset_name STRING COMMENT 'Bitcoin, Ethereum ...; unknown for symbols without a name mapping',
    CONSTRAINT pk_dim_asset PRIMARY KEY (asset_key)
)
COMMENT 'Crypto asset dimension (SCD1), one row per symbol'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD}.dim_source (
    source_key  BIGINT GENERATED BY DEFAULT AS IDENTITY,
    source_name STRING NOT NULL COMMENT 'Value of `source` in silver: kraken_api, streaming',
    source_type STRING COMMENT 'batch / streaming',
    provider    STRING COMMENT 'Kraken / CoinGecko via Event Hub',
    CONSTRAINT pk_dim_source PRIMARY KEY (source_key)
)
COMMENT 'Data source dimension'
""")

### Facts

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD}.fact_ohlc_hourly (
    asset_key BIGINT NOT NULL COMMENT 'Asset of the event, key of dim_asset',
    date_key         INT       NOT NULL,
    source_key       BIGINT    NOT NULL,
    event_time       TIMESTAMP NOT NULL COMMENT 'Start of the hourly candle (UTC)',
    hour_of_day      INT,
    open             DOUBLE,
    high             DOUBLE,
    low              DOUBLE,
    close            DOUBLE    COMMENT 'Non-additive: use avg/min/max/last, never SUM',
    vwap             DOUBLE,
    volume           DOUBLE    COMMENT 'Traded amount in COINS for this hour (additive)',
    trade_count      BIGINT,
    price_change_pct DOUBLE    COMMENT '(close - open) / open * 100',
    processed_at     TIMESTAMP,
    CONSTRAINT pk_fact_ohlc_hourly PRIMARY KEY (asset_key, event_time),
    CONSTRAINT fk_ohlc_asset  FOREIGN KEY (asset_key)  REFERENCES {GOLD}.dim_asset (asset_key),
    CONSTRAINT fk_ohlc_date   FOREIGN KEY (date_key)   REFERENCES {GOLD}.dim_date (date_key),
    CONSTRAINT fk_ohlc_source FOREIGN KEY (source_key) REFERENCES {GOLD}.dim_source (source_key)
)
CLUSTER BY (date_key)
COMMENT 'Hourly OHLC candles from Kraken (batch). Grain: one asset x one hour'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD}.fact_price_tick (
    asset_key BIGINT NOT NULL COMMENT 'Asset of the event, key of dim_asset',
    date_key     INT       NOT NULL,
    source_key   BIGINT    NOT NULL,
    event_time   TIMESTAMP NOT NULL,
    price_usd    DOUBLE    COMMENT 'Non-additive: use avg/min/max/last, never SUM',
    volume_24h   DOUBLE    COMMENT 'Rolling 24h volume in USD; NOT additive across ticks, NOT comparable with fact_ohlc_hourly.volume',
    processed_at TIMESTAMP,
    CONSTRAINT pk_fact_price_tick PRIMARY KEY (asset_key, event_time),
    CONSTRAINT fk_tick_asset  FOREIGN KEY (asset_key)  REFERENCES {GOLD}.dim_asset (asset_key),
    CONSTRAINT fk_tick_date   FOREIGN KEY (date_key)   REFERENCES {GOLD}.dim_date (date_key),
    CONSTRAINT fk_tick_source FOREIGN KEY (source_key) REFERENCES {GOLD}.dim_source (source_key)
)
CLUSTER BY (date_key)
COMMENT 'Live price ticks from CoinGecko via Lakeflow. Grain: one asset x one moment'
""")